# Module 05 — Data Locality, Shuffle & SPJ

In [1]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs

spark = begin_module('module-05-data-locality')
ui_links()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/17 12:56:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

## Why shuffle appears

In [2]:
from pyspark.sql import functions as F, Window
trips = spark.read.parquet(*config.yellow_taxi_paths())
agg = trips.groupBy('PULocationID', 'VendorID').count()
agg.explain('formatted')  # Exchange
w = Window.partitionBy('VendorID').orderBy(F.desc('total_amount'))
win = trips.withColumn('rk', F.row_number().over(w))
win.limit(1).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [VendorID#0, PULocationID#7]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
ReadSchema: struct<VendorID:int,PULocationID:int>

(2) HashAggregate
Input [2]: [VendorID#0, PULocationID#7]
Keys [2]: [PULocationID#7, VendorID#0]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#41L]
Results [3]: [PULocationID#7, VendorID#0, count#42L]

(3) Exchange
Input [3]: [PULocationID#7, VendorID#0, count#42L]
Arguments: hashpartitioning(PULocationID#7, VendorID#0, 200), ENSURE_REQUIREMENTS, [plan_id=11]

(4) HashAggregate
Input [3]: [PULocationID#7, VendorID#0, count#42L]
Keys [2]: [PULocationID#7, VendorID#0]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#39L]
Results [3]: [PULocationID#7, VendorID#0, count(1)#39L AS count#19L]

(5

## Join hints

In [4]:
zones = spark.read.option('header', True).csv(config.zones_path())
z = zones.withColumnRenamed('LocationID', 'PULocationID')
from pyspark.sql.functions import broadcast
hinted = trips.join(broadcast(z), 'PULocationID', 'inner')
hinted.limit(1).explain('formatted')

== Physical Plan ==
AdaptiveSparkPlan (10)
+- CollectLimit (9)
   +- Project (8)
      +- BroadcastHashJoin Inner BuildRight (7)
         :- Filter (2)
         :  +- Scan parquet  (1)
         +- BroadcastExchange (6)
            +- Project (5)
               +- Filter (4)
                  +- Scan csv  (3)


(1) Scan parquet 
Output [19]: [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/cache/yellow_tripdata_2024-01.parquet, ... 1 entries]
PushedFilters: [IsNotNull(PULocationID)]
ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passenger_count:bigint,trip_distance:double,RatecodeID:b

## Storage Partitioned Join — setup

In [5]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'true')
spark.conf.set('spark.sql.sources.v2.bucketing.pushPartValues.enabled', 'true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping', 'true')
spark.conf.set('spark.sql.requireAllClusterKeysForCoPartition', 'false')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

base = trips.select('VendorID', 'PULocationID', 'total_amount')
drop_iceberg_table(spark, 'local.default.spj_a')
drop_iceberg_table(spark, 'local.default.spj_b')
base.writeTo('local.default.spj_a').using('iceberg').partitionedBy('PULocationID').create()
base.writeTo('local.default.spj_b').using('iceberg').partitionedBy('PULocationID').create()

### Vanilla SPJ

In [6]:
spj = spark.sql('''SELECT a.VendorID, a.total_amount, b.total_amount AS amt_b
    FROM local.default.spj_a a JOIN local.default.spj_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
spj.explain('formatted')  # no Exchange before join = success

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin Inner (7)
      :- Sort (3)
      :  +- Filter (2)
      :     +- BatchScan local.default.spj_a (1)
      +- Sort (6)
         +- Filter (5)
            +- BatchScan local.default.spj_b (4)


(1) BatchScan local.default.spj_a
Output [3]: [VendorID#98, PULocationID#99, total_amount#100]
IcebergScan(table=local.default.spj_a, schemaId=0, snapshotId=4792744476913068324, branch=null, filters=PULocationID IS NOT NULL, VendorID IS NOT NULL, runtimeFilters=, groupedBy=PULocationID)

(2) Filter
Input [3]: [VendorID#98, PULocationID#99, total_amount#100]
Condition : isnotnull(VendorID#98)

(3) Sort
Input [3]: [VendorID#98, PULocationID#99, total_amount#100]
Arguments: [PULocationID#99 ASC NULLS FIRST, VendorID#98 ASC NULLS FIRST], false, 0

(4) BatchScan local.default.spj_b
Output [3]: [VendorID#101, PULocationID#102, total_amount#103]
IcebergScan(table=local.default.spj_b, schemaId=0, snapshotId=142046193652303608, br

### SPJ off — Exchange returns

In [7]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'false')
no_spj = spark.sql('''SELECT a.VendorID FROM local.default.spj_a a JOIN local.default.spj_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
no_spj.explain('formatted')
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'true')

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- BatchScan local.default.spj_a (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- BatchScan local.default.spj_b (5)


(1) BatchScan local.default.spj_a
Output [2]: [VendorID#120, PULocationID#121]
IcebergScan(table=local.default.spj_a, schemaId=0, snapshotId=4792744476913068324, branch=null, filters=PULocationID IS NOT NULL, VendorID IS NOT NULL, runtimeFilters=, groupedBy=PULocationID)

(2) Filter
Input [2]: [VendorID#120, PULocationID#121]
Condition : isnotnull(VendorID#120)

(3) Exchange
Input [2]: [VendorID#120, PULocationID#121]
Arguments: hashpartitioning(PULocationID#121, VendorID#120, 200), ENSURE_REQUIREMENTS, [plan_id=301]

(4) Sort
Input [2]: [VendorID#120, PULocationID#121]
Arguments: [PULocationID#121 ASC NULLS FIRST, VendorID#120 ASC NULLS FIRST], false, 0

(5)

### One-sided SPJ (only one table partition-aligned)

Join Iceberg partitioned table to ad-hoc Parquet — expect shuffle on the unclustered side.

In [8]:
side_path = str(config.OUTPUT_BASE / '05-side.parquet')
base.write.mode('overwrite').parquet(side_path)
side = spark.read.parquet(side_path)
mixed = spark.table('local.default.spj_a').join(side, ['PULocationID', 'VendorID'])
mixed.limit(1).explain('formatted')

[Stage 8:====================================================>    (11 + 1) / 12]

== Physical Plan ==
AdaptiveSparkPlan (12)
+- CollectLimit (11)
   +- Project (10)
      +- SortMergeJoin Inner (9)
         :- Sort (4)
         :  +- Exchange (3)
         :     +- Filter (2)
         :        +- BatchScan local.default.spj_a (1)
         +- Sort (8)
            +- Exchange (7)
               +- Filter (6)
                  +- Scan parquet  (5)


(1) BatchScan local.default.spj_a
Output [3]: [VendorID#143, PULocationID#144, total_amount#145]
IcebergScan(table=local.default.spj_a, schemaId=0, snapshotId=4792744476913068324, branch=null, filters=PULocationID IS NOT NULL, VendorID IS NOT NULL, runtimeFilters=, groupedBy=PULocationID)

(2) Filter
Input [3]: [VendorID#143, PULocationID#144, total_amount#145]
Condition : isnotnull(VendorID#143)

(3) Exchange
Input [3]: [VendorID#143, PULocationID#144, total_amount#145]
Arguments: hashpartitioning(PULocationID#144, VendorID#143, 200), ENSURE_REQUIREMENTS, [plan_id=359]

(4) Sort
Input [3]: [VendorID#143, PULocationID#144, t